# Open-CD — Introduction and Installation Verification

[Open-CD](https://github.com/likyoo/open-cd) is the most complete open-source library for **change detection** in remote sensing imagery. It provides 20+ state-of-the-art models, datasets, and evaluation tools in a unified config-driven framework built on OpenMMLab.

## The OpenMMLab Ecosystem

Open-CD builds on these OpenMMLab components:

| Package | Role |
|---|---|
| `mmengine` | Config system, training runner, logging |
| `mmcv` | Image/video processing ops, CUDA-accelerated layers |
| `mmsegmentation` | Segmentation model architectures and datasets |
| `mmdet` | Detection utilities (used by some CD models) |
| `open-cd` | Change detection models, datasets, metrics |

The config system is central to OpenMMLab: each training run is defined by a Python config file that specifies the model, dataset, scheduler, and optimizer. This makes experiments fully reproducible.

## References
- Open-CD: https://github.com/likyoo/open-cd
- OpenMMLab: https://openmmlab.com/
- Open-CD paper: https://arxiv.org/abs/2407.15317

## 1. Verify Installation

In [ ]:
import torch
import mmengine
import mmcv
import mmseg
import opencd

print(f"PyTorch:        {torch.__version__}")
print(f"mmengine:       {mmengine.__version__}")
print(f"mmcv:           {mmcv.__version__}")
print(f"mmsegmentation: {mmseg.__version__}")
print(f"open-cd:        {opencd.__version__}")
print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version:   {torch.version.cuda}")
    print(f"GPU:            {torch.cuda.get_device_name(0)}")

## 2. GPU mmcv Setup (if needed)

The default PyPI `mmcv` wheel provides CPU-only ops. GPU ops (deformable conv, etc.) require a CUDA-compiled wheel. Run this cell to get the correct install command for your environment.

In [ ]:
import torch
import mmcv

if torch.cuda.is_available():
    cuda_version = torch.version.cuda.replace(".", "")  # e.g., "11.8" -> "118"
    torch_major = torch.__version__.split("+")[0].rsplit(".", 1)[0]  # e.g., "2.1"
    cuda_key = f"cu{cuda_version}"
    torch_key = f"torch{torch_major}"
    url = f"https://download.openmmlab.com/mmcv/dist/{cuda_key}/{torch_key}/index.html"

    print("To install GPU-accelerated mmcv, run:")
    print(f"""\nuv run pip install mmcv -f {url}""")
    print("\nThen restart the kernel.")
else:
    print("No GPU detected. CPU-only mmcv is sufficient.")
    print("GPU ops (deformable conv) will fall back to CPU implementations.")

## 3. Available Change Detection Models

In [ ]:
from opencd.registry import MODELS

# List registered change detection model architectures
cd_models = [name for name in MODELS._module_dict.keys()]
print(f"Registered models in Open-CD: {len(cd_models)}")
print("\nSample models:")
for m in sorted(cd_models)[:15]:
    print(f"  - {m}")
if len(cd_models) > 15:
    print(f"  ... and {len(cd_models) - 15} more")

## 4. The Config System

Open-CD uses mmengine's config system. Each config file inherits from a base config and overrides specific fields. This is how you switch models, datasets, or hyperparameters.

In [ ]:
import os
import opencd

# Find the Open-CD configs directory
opencd_dir = os.path.dirname(opencd.__file__)
configs_root = os.path.join(opencd_dir, "..", "configs")

if os.path.exists(configs_root):
    model_dirs = [d for d in os.listdir(configs_root) if os.path.isdir(os.path.join(configs_root, d))]
    print("Available model config directories:")
    for d in sorted(model_dirs):
        print(f"  configs/{d}/")
else:
    print(f"Configs not found at {configs_root}")
    print("Configs are in the source repo: https://github.com/likyoo/open-cd/tree/main/configs")

## 5. Change Detection Model Taxonomy

Open-CD organizes models by their fusion strategy:

| Strategy | Examples | Approach |
|---|---|---|
| Early fusion | FC-EF | Concatenate T1+T2 as input |
| Siamese | SNUNet, ChangeFormer | Process T1 and T2 separately, compare features |
| Pseudo-change | TTP | Use pseudo-labels from unlabeled data |
| Foundation model | BAN, Changer | Fine-tune a pretrained backbone |
| Semantic CD | Changer-s | Predict *from* and *to* class for each changed pixel |

**Recommended starting point:** `Changer-r18` — good accuracy, fast training, well-supported in Open-CD.

Continue to `1-levir_cd_dataset.ipynb` to explore the standard benchmark dataset.